# 현금 등 기타 지출 내역

In [4]:
from ledgerly.expenditure import (shinhan_file_config
                                  , map_shinhan_card_df_to_expenditure
                                  , preprocess_shinhan_data
                                  , map_category
                                  , insert_expenditure_data
                                  , fetch_expenditure_data)
from ledgerly.utils import find_project_root

In [5]:
import pandas as pd

## 1. 현금 파일로부터 데이터 불러오기

In [6]:
cash_df = pd.read_csv(find_project_root() / "data" / "raw" / "cash_form.csv")

In [7]:
cash_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   이용일     15 non-null     object 
 1   사용처     15 non-null     object 
 2   금액      15 non-null     int64  
 3   결제수단    15 non-null     object 
 4   결제제공자   15 non-null     object 
 5   memo    0 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 852.0+ bytes


In [8]:
cash_df.head()

,이용일,사용처,금액,결제수단,결제제공자,memo
0,2026-01-31,경조사(용수),100000,현금,KB,NaN
1,2026-01-27,맑은하늘적금,200000,현금,KB,NaN
2,2026-01-21,DB손해보험,80330,현금,KB,NaN
3,2026-01-21,운전자보험,13130,현금,KB,NaN
4,2026-01-09,어린이보험,73000,현금,KB,NaN


## 2. 현금 데이터 전처리

In [11]:
# cols = ["이용일","사용처","금액","결제수단","결제제공자","memo"]

def preprocess_cash_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    #날짜 형식 변환
    df["이용일"] = pd.to_datetime(df["이용일"], format="%Y-%m-%d")

    #텍스트 컬럼 변경
    text_cols = ["사용처", "결제수단", "결제제공자", "memo"]
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()

    #금액 컬럼 변경
    df["금액"] = pd.to_numeric(df["금액"], errors="coerce").fillna(0).astype(int)

    return df

In [12]:
cash_df = preprocess_cash_data(cash_df)
cash_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   이용일     15 non-null     datetime64[ns]
 1   사용처     15 non-null     object        
 2   금액      15 non-null     int64         
 3   결제수단    15 non-null     object        
 4   결제제공자   15 non-null     object        
 5   memo    15 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 852.0+ bytes


In [13]:
cash_df.head()

,이용일,사용처,금액,결제수단,결제제공자,memo
0,2026-01-31,경조사(용수),100000,현금,KB,nan
1,2026-01-27,맑은하늘적금,200000,현금,KB,nan
2,2026-01-21,DB손해보험,80330,현금,KB,nan
3,2026-01-21,운전자보험,13130,현금,KB,nan
4,2026-01-09,어린이보험,73000,현금,KB,nan


In [14]:
def map_cash_df_to_expenditure(df: pd.DataFrame) -> pd.DataFrame:
    mapped_df = pd.DataFrame()
    mapped_df["used_at"] = df["이용일"]
    mapped_df["merchant_name"] = df["사용처"]
    mapped_df["amount"] = df["금액"]
    mapped_df["payment_type"] = df["결제수단"]
    mapped_df["payment_provider"] = df["결제제공자"]
    mapped_df["memo"] = df["memo"]

    mapped_df["category"] = "unknown"
    mapped_df["installment_type"] = "single"


    mapped_df["source_uid"] = (
    mapped_df["payment_type"]
    + "_"
    + mapped_df["payment_provider"]
    + "_" + mapped_df["used_at"].dt.strftime("%Y%m%d")
    + "_" + mapped_df["merchant_name"]
    + "_" + mapped_df["memo"]
    )

    return mapped_df

In [15]:
expenditure_df = map_cash_df_to_expenditure(cash_df)
expenditure_df.head()

,used_at,merchant_name,amount,payment_type,payment_provider,memo,category,installment_type,source_uid
0,2026-01-31,경조사(용수),100000,현금,KB,nan,unknown,single,현금_KB_20260131_경조사(용수)_nan
1,2026-01-27,맑은하늘적금,200000,현금,KB,nan,unknown,single,현금_KB_20260127_맑은하늘적금_nan
2,2026-01-21,DB손해보험,80330,현금,KB,nan,unknown,single,현금_KB_20260121_DB손해보험_nan
3,2026-01-21,운전자보험,13130,현금,KB,nan,unknown,single,현금_KB_20260121_운전자보험_nan
4,2026-01-09,어린이보험,73000,현금,KB,nan,unknown,single,현금_KB_20260109_어린이보험_nan


In [17]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df.head()

,used_at,merchant_name,amount,payment_type,payment_provider,memo,category,installment_type,source_uid
0,2026-01-31,경조사(용수),100000,현금,KB,nan,경조사,single,현금_KB_20260131_경조사(용수)_nan
1,2026-01-27,맑은하늘적금,200000,현금,KB,nan,저축,single,현금_KB_20260127_맑은하늘적금_nan
2,2026-01-21,DB손해보험,80330,현금,KB,nan,보험,single,현금_KB_20260121_DB손해보험_nan
3,2026-01-21,운전자보험,13130,현금,KB,nan,보험,single,현금_KB_20260121_운전자보험_nan
4,2026-01-09,어린이보험,73000,현금,KB,nan,보험,single,현금_KB_20260109_어린이보험_nan


In [18]:
insert_expenditure_data(expenditure_df)
fetch_expenditure_data()

,id,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid,created_at
0,1,2026-01-03,card,kb,쿠팡(쿠페이),single,32600,0,쇼핑,None,card_kb_30118172,2026-01-25 09:28:28
1,2,2026-01-03,card,kb,네이버페이,single,24840,0,쇼핑,None,card_kb_30118161,2026-01-25 09:28:28
2,3,2026-01-03,card,kb,네이버페이,single,47500,0,쇼핑,None,card_kb_30118150,2026-01-25 09:28:28
3,4,2026-01-03,card,kb,(주)딜리유통,single,42000,0,기타,None,card_kb_30118141,2026-01-25 09:28:28
4,5,2026-01-03,card,kb,웨이브(푹TV)자동,single,10900,0,문화,None,card_kb_30118138,2026-01-25 09:28:28
5,6,2026-01-01,card,kb,NICE 결제대행_4,single,110000,0,기타,None,card_kb_30118063,2026-01-25 09:28:28
6,7,2026-01-01,card,kb,구글플레이,single,19784,0,문화,None,card_kb_30118057,2026-01-25 09:28:28
7,8,2026-01-24,card,shinhan,홈플러스㈜,single,68210,0,식비,None,card_shinhan_25863165,2026-01-28 12:37:02
8,9,2026-01-16,card,shinhan,홈플러스㈜,single,50240,0,식비,None,card_shinhan_31299218,2026-01-28 12:37:02
9,10,2026-01-16,card,shinhan,(주)아트박스 강서 홈플러스점,single,2000,0,식비,None,card_shinhan_23635058,2026-01-28 12:37:02
